# FSD AI vs Real Image Detection (Kaggle)
This notebook runs the Forensic Self-Descriptions (FSD) detector on a Kaggle dataset or a single uploaded image.

## What you need to do:
1) Turn **Internet ON** in Kaggle Notebook settings
2) Select a **GPU** accelerator (T4 or P100)
3) Add the dataset in Kaggle (so it appears under `/kaggle/input/ai_image_dataset/`)
4) Run cells in order

In [1]:
# Clone repo and install dependencies
!git clone https://github.com/ductai199x/Forensic-Self-Descriptions-CVPR25.git
!pip -q install uv
!cd Forensic-Self-Descriptions-CVPR25 && uv pip install --system -e .

Cloning into 'Forensic-Self-Descriptions-CVPR25'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 139 (delta 49), reused 54 (delta 23), pack-reused 50 (from 1)
Receiving objects: 100% (139/139), 51.13 MiB | 41.13 MiB/s, done.
Resolving deltas: 100% (69/69), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 79.1 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 127 packages in 1.06s
   Building fsd-detector @ file:///kaggle/working/Forensic-Self-Descriptions-CVP
   Building fsd-detector @ file:///kaggle/working/Forensic-Self-Descriptions-CVP
⠙ Preparing packages... (0/22)
   Building fsd-detector @ file:///kaggle/working/Forensic-Self-Descriptions-CVP
⠙ Preparing packages... (0/22)
   Building fsd-detector @ file:///kaggle/working/Forensic-Self-Descriptions-CVP
⠙ Preparing packages... (0/22)
opentelemetry-exporter-prometheus ------------------------------

## Evaluate on the Dataset
This cell reads images from `/kaggle/input/ai_image_dataset/benchmark-v0.1/images/` (ai vs real), scores them, and prints accuracy, precision, recall, and F1.

## Quick EDA (before evaluation)
Counts total images, class balance, and per-category counts under `images/ai` and `images/real`.

In [2]:
from pathlib import Path
from collections import Counter

mounted_root = Path("/kaggle/input/datasets/kmazd1110/ai-image-dataset/benchmark-v0.1/images")
downloaded_root = Path("/kaggle/working/benchmark-v0.1/images")
dataset_root = mounted_root if mounted_root.exists() else downloaded_root
assert dataset_root.exists(), "Dataset not found. Add the dataset in Kaggle or run the download cell."

exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
class_counts = Counter()
category_counts = Counter()

for class_dir in ["ai", "real"]:
    base = dataset_root / class_dir
    if not base.exists():
        continue
    for p in base.rglob("*"):
        if p.suffix.lower() in exts:
            class_counts[class_dir] += 1
            # category is the first folder under class (ai/animal, real/portrait, etc.)
            parts = p.relative_to(base).parts
            category = parts[0] if len(parts) > 1 else "(uncategorized)"
            category_counts[(class_dir, category)] += 1

total = sum(class_counts.values())
print(f"Dataset root: {dataset_root}")
print(f"Total images: {total}")
print(f"AI images: {class_counts.get('ai', 0)}")
print(f"Real images: {class_counts.get('real', 0)}")
print("\nTop categories by class:")
for (cls, cat), count in category_counts.most_common(20):
    print(f"{cls}/{cat}: {count}")

Dataset root: /kaggle/input/datasets/kmazd1110/ai-image-dataset/benchmark-v0.1/images
Total images: 2038
AI images: 1018
Real images: 1020

Top categories by class:
ai/art: 170
ai/animal: 170
ai/product: 170
ai/portrait: 170
ai/food: 170
real/art: 170
real/animal: 170
real/product: 170
real/portrait: 170
real/food: 170
real/landscape: 170
ai/landscape: 168


In [3]:
import os
import sys
import random
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Add repo to path
repo_path = "/kaggle/working/Forensic-Self-Descriptions-CVPR25"
if repo_path not in sys.path:
    sys.path.append(repo_path)

from fsd import FSDDetector

mounted_root = Path("/kaggle/input/datasets/kmazd1110/ai-image-dataset/benchmark-v0.1/images")
downloaded_root = Path("/kaggle/input/datasets/kmazd1110/ai-image-dataset/benchmark-v0.1/images")
dataset_root = mounted_root if mounted_root.exists() else downloaded_root
assert dataset_root.exists(), "Dataset not found. Add the dataset in Kaggle or run the download cell."

samples_per_category = 60
random_seed = 42


def build_from_dirs(root: Path, samples_per_cat: int, seed: int):
    exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    rng = random.Random(seed)
    paths = []
    labels = []

    for label_dir, label in [("real", 0), ("ai", 1)]:
        base = root / label_dir
        if not base.exists():
            continue

        category_dirs = [p for p in base.iterdir() if p.is_dir()]
        if not category_dirs:
            category_dirs = [base]

        for cat_dir in category_dirs:
            candidates = [p for p in cat_dir.rglob("*") if p.suffix.lower() in exts]
            if not candidates:
                continue
            take = min(samples_per_cat, len(candidates))
            sampled = rng.sample(candidates, take)
            paths.extend([str(p) for p in sampled])
            labels.extend([label] * take)

    if not paths:
        raise ValueError("No labeled images found under images/ai and images/real.")
    return paths, labels


image_paths, y_true = build_from_dirs(dataset_root, samples_per_category, random_seed)

print(f"Scoring {len(image_paths)} images ({samples_per_category} per category, per class)...")
detector = FSDDetector.load(attribution=False)
results = detector.score_batch(image_paths)
y_pred = [1 if r.is_fake else 0 for r in results]

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print("\n--- Overall Metrics ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")

print("\n--- Per-Model Metrics ---")
model_names = []
for p, y in zip(image_paths, y_true):
    if y == 1:
        cat = Path(p).parent.name
        stem = Path(p).stem
        if f"_{cat}_" in stem:
            model_names.append(stem.split(f"_{cat}_")[0])
        else:
            model_names.append("unknown_model")
    else:
        model_names.append("real")

ai_models = sorted(list(set(m for m, y in zip(model_names, y_true) if y == 1)))

for model in ai_models:
    indices = [i for i, m in enumerate(model_names) if m == model or m == "real"]
    y_t = [y_true[i] for i in indices]
    y_p = [y_pred[i] for i in indices]
    
    if len(set(y_t)) > 1:
        m_acc = accuracy_score(y_t, y_p)
        m_prec = precision_score(y_t, y_p, zero_division=0)
        m_rec = recall_score(y_t, y_p, zero_division=0)
        m_f1 = f1_score(y_t, y_p, zero_division=0)
        
        print(f"\nModel: {model}")
        print(f"  Accuracy:  {m_acc:.4f}")
        print(f"  Precision: {m_prec:.4f}")
        print(f"  Recall:    {m_rec:.4f}")
        print(f"  F1:        {m_f1:.4f}")

Scoring 720 images (60 per category, per class)...


Scoring: 100%|██████████| 720/720 [2:33:43<00:00, 12.81s/img]


--- Overall Metrics ---
Accuracy:  0.8819
Precision: 0.8686
Recall:    0.9000
F1:        0.8840

--- Per-Model Metrics ---

Model: flux_2_flex
  Accuracy:  0.8707
  Precision: 0.2794
  Recall:    1.0000
  F1:        0.4368

Model: flux_pro_v1.1
  Accuracy:  0.8721
  Precision: 0.3797
  Recall:    0.9677
  F1:        0.5455

Model: flux_schnell
  Accuracy:  0.8711
  Precision: 0.2899
  Recall:    1.0000
  F1:        0.4494

Model: gemini_3_pro
  Accuracy:  0.8714
  Precision: 0.3000
  Recall:    1.0000
  F1:        0.4615

Model: glm_image
  Accuracy:  0.8717
  Precision: 0.3099
  Recall:    1.0000
  F1:        0.4731

Model: gpt_image_1.5
  Accuracy:  0.8724
  Precision: 0.3288
  Recall:    1.0000
  F1:        0.4948

Model: grok_aurora
  Accuracy:  0.8717
  Precision: 0.3099
  Recall:    1.0000
  F1:        0.4731

Model: hunyuan_v3
  Accuracy:  0.8693
  Precision: 0.2344
  Recall:    1.0000
  F1:        0.3797

Model: ideogram_v3
  Accuracy:  0.8697
  Precision: 0.2462
  Recall:    